In [2]:
from langchain_openai import ChatOpenAI
import os 
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool

model = ChatOpenAI(
    model = "nvidia/nemotron-3-super-120b-a12b:free",
    api_key = os.environ["OPENROUTER_API_KEY"],
    base_url = "https://openrouter.ai/api/v1"
)
@tool
def search_hotels(city:str) -> str:
    """Search hotels : returns long response to use more tokens."""
    return f""" Hotels in {city}:
    1. Grand Hotel - 5 star, ₹350/night, spa, pool, gym
    2. City Inn - 4 star, ₹100/night, business center
    3. Budget Stay - 3 star, ₹75/night, free wifi
"""

agent = create_agent(
    model = model,
    tools = [search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model = model,
            trigger = ("tokens",550),
            keep=("tokens",200)
        )
    ]
)

agent
config = {"configurable" : {"thread_id" : "test_1"}}

##Token counter(approximate)

def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4 # 4 chars = 1 token

In [3]:
## Run test
cities = ["Paris", "London","Tokyo", "New York","Dubai","Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages" : [HumanMessage(content = f"find hotels in {city}")]},
        config = config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city} : ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris : ~168 tokens, 4 messages
[HumanMessage(content='find hotels in Paris', additional_kwargs={}, response_metadata={}, id='7263e8c5-7109-4e57-8281-356ba8dee21f'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 73, 'prompt_tokens': 277, 'total_tokens': 350, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 46, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}}, 'model_provider': 'openai', 'model_name': 'nvidia/nemotron-3-super-120b-a12b-20230311:free', 'system_fingerprint': None, 'id': 'gen-1782884331-kdS263rTZrJ3imnlNeuG', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f1c2f-eca

### Human In the loop Middleware